# Project 04 — BROKEN notebook (debugging exercise)

This notebook contains **seeded bugs** centred on the project's pitfall: un-scaled predictors and the priors they break. Run it, read the diagnostics, find each bug, and fix it. The clean reference is `notebook.ipynb`; the answer key is `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240604

In [ ]:
from data.generate_data import generate
data = generate()
x, y = data['x'], data['y']   # NOTE: raw dose, NOT standardized

### Model — raw predictor with a slope prior meant for a standardized scale.

In [ ]:
# BUG 1 (headline): regress on RAW x (doses ~100..600), not standardized x.
#   The intercept becomes an extrapolation to x=0 and alpha/beta correlate ~1.
# BUG 2: a too-tight slope prior Normal(0, 0.5) that is fine on the
#   standardized scale but absurd on the raw scale, where the true slope is
#   ~0.015. (Here it is not too tight; the danger is using the SAME width on
#   raw and standardized scales without thinking -- see BROKEN_BUGS.md.)
with pm.Model() as model:
    alpha = pm.Normal('alpha', mu=0.0, sigma=0.5)   # BUG 2: width copied from std scale
    beta = pm.Normal('beta', mu=0.0, sigma=0.5)
    sigma = pm.HalfNormal('sigma', sigma=2.0)
    mu = alpha + beta * x                            # BUG 1: raw x
    pm.Normal('y', mu=mu, sigma=sigma, observed=y)
    idata = pm.sample(draws=1000, tune=1000, chains=2, random_seed=RNG,
                      progressbar=False)

In [ ]:
print(az.summary(idata, var_names=['alpha', 'beta', 'sigma']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))
print('natural-scale truth:', data['truth_natural'])

### Posterior predictive overlay — BUG 3: line plotted on the wrong x scale.

In [ ]:
a = idata.posterior['alpha'].values.ravel()
b = idata.posterior['beta'].values.ravel()
# BUG 3: the fit used RAW x, but here we overlay the line on STANDARDIZED x,
#   so the line will not match the data cloud.
xs = np.linspace(data['x_std'].min(), data['x_std'].max(), 50)
fig, ax = plt.subplots(figsize=(6,3.5))
ax.scatter(x, y, color='#4C72B0', label='data (raw x)')
ax.plot(xs, a.mean() + b.mean()*xs, 'k-', label='fitted line (std x) -- mismatched!')
ax.legend(); plt.tight_layout()